In [ ]:
directory = 'AlzheimersFLAIR/data-preprocessing/n4-corrected2'

#out_directory = 'AlzheimersFLAIR/2D-models/largest-ventricles2'

problematic_registers = []
# patient ids removed for privacy purposes

selected_volumes = []


for s in os.listdir(directory):
    if (s != '.DS_Store'):
        if s in problematic_registers:
            print("found problematic register, skipping...")
            continue
        
        filepath = os.path.join(directory, s)
        img = np.array([contrast_stretch(i) * 255 for i in nib.load(filepath).get_fdata().transpose(2, 1, 0)]).astype(np.uint8)[12:28]
        img = np.array([cv2.resize(i, target_size, interpolation = cv2.INTER_LINEAR) for i in img])
        img = np.repeat(np.expand_dims(img, -1), 3, axis = -1)
        
        out_array = contrast_stretch(nib.load(filepath).get_fdata().transpose(2, 0, 1))[12:28]
        
        print(np.shape(img))
        masks = model.predict(img)
        print(np.shape(masks))
        
        largest_index = 0
        for i in range(len(masks)):
            current_area = np.sum(masks[i])
            largest_area = np.sum(masks[largest_index])
            if (current_area > largest_area):
                largest_index = i
        
        jpg_path = out_directory
        patient_id = s[:10]
        # print(patient_id)  # removed for privacy
        diagnosis = labels.loc[labels['Subject ID'] == patient_id, 'Research Group'].values[0]
        if (diagnosis == 'MCI'): continue
        if (diagnosis == 'EMCI'):
            jpg_path = os.path.join(jpg_path, '1')
        elif (diagnosis == 'LMCI'):
            jpg_path = os.path.join(jpg_path, '2')
        elif (diagnosis == 'AD'):
            jpg_path = os.path.join(jpg_path, '3')
        else:
            jpg_path = os.path.join(jpg_path, '0')
        jpg_path = os.path.join(jpg_path, jpg_name)
        out_mask = Image.fromarray(np.rot90((out_array[largest_index - 5] * 255).astype(np.uint8), k = 1))
        out_mask.save(jpg_path)
        